# 6. Full Transformers — T5, FLAN-T5 i BART

Notebook zamyka przegląd trzech głównych rodzin architektur transformatorowych:

- **encoder**,
- **decoder**,
- **encoder-decoder**.

W architekturze encoder-decoder encoder buduje reprezentację wejścia, a decoder generuje wyjście krok po kroku z użyciem **cross-attention**.  
Taki układ pozostaje praktyczny także w aktualnie, szczególnie wtedy, gdy potrzebna jest kontrolowana transformacja tekstu:

- summarization,
- translation,
- paraphrase,
- generowanie krótkich odpowiedzi na podstawie wejścia,
- ekstrakcja i normalizacja informacji.

Aktualnie ogólne systemy chatowe są najczęściej budowane na modelach **decoder-only**, ale rodzina **seq2seq** nadal pozostaje bardzo ważna tam, gdzie liczy się czytelna relacja **wejście → wyjście**, niski lub średni koszt inferencji albo model wyspecjalizowany w zadaniu.



## Wymagania

Przydatne elementy środowiska:

- konto w serwisie Hugging Face, gdyby model wymagał potwierdzenia warunków użycia lub pojawiły się limity pobrań,
- środowisko typu Colab / Jupyter,
- opcjonalnie GPU, aby przyspieszyć inferencję.

Pakiety używane w notebooku:

- `transformers` – modele, tokenizery i generowanie tekstu,
- `datasets` – wygodna praca na zbiorach danych,
- `evaluate` – prosta ewaluacja metrykami,
- `sentencepiece` – wymagane m.in. przez rodzinę T5,
- `accelerate` – wygodniejsze uruchamianie modeli.

Notebook koncentruje się na **inferencji i porównywaniu modeli**, a nie na pełnym fine-tuningu.

Wersja materiału została dopasowana do zmian w `transformers` 5.x, gdzie historyczne pipeline’y `summarization` i `text2text-generation` nie są już bezpiecznym punktem odniesienia dla nowych notebooków. Z tego powodu demonstracje oparto na bezpośrednim użyciu `AutoTokenizer`, `AutoModelForSeq2SeqLM` i `generate()`.


In [5]:
!pip -q install -U "transformers>=5.0.0" datasets evaluate sentencepiece accelerate

## Krótkie porównanie: BERT vs GPT vs T5 / BART

| Typ modelu | Architektura | Główna rola | Typowe zadania | Typowa klasa | Miejsce w praktyce |
|---|---|---|---|---|---|
| BERT | Encoder | zrozumienie wejścia | klasyfikacja, NER, embeddingi, reranking, fill-mask | `AutoModel`, `AutoModelForMaskedLM` | nadal bardzo ważny w zadaniach rozumienia tekstu i retrievalu |
| GPT i podobne | Decoder | generowanie kolejnych tokenów | chat, completion, agenty, kod, generacja otwarta | `AutoModelForCausalLM` | dominują w ogólnych asystentach i systemach instrukcyjnych |
| T5 / FLAN-T5 / BART | Encoder-Decoder | transformacja wejścia w wyjście | summarization, translation, paraphrase, kontrolowany text-to-text | `AutoModelForSeq2SeqLM` | nadal bardzo przydatne w zadaniach kontrolowanych i modelach wyspecjalizowanych |

Najważniejsza intuicja:

- **BERT** buduje reprezentację wejścia i zwykle nie generuje pełnego nowego tekstu.
- **GPT** generuje tekst od lewej do prawej.
- **T5 / BART** najpierw czytają wejście, a następnie generują wynik na jego podstawie.

To właśnie dlatego modele seq2seq szczególnie dobrze pasują do zadań, w których wejście i wyjście są tekstem, ale celem nie jest zwykła kontynuacja promptu, tylko **przekształcenie jednego tekstu w drugi**.


## Import wymaganych bibliotek

In [6]:
import torch
import pandas as pd
import evaluate
import transformers

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    set_seed
)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Transformers version:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Urządzenie:", device)


Transformers version: 5.4.0
CUDA available: True
Urządzenie: cuda


## Rodzina seq2seq

W praktyce rodzina encoder-decoder jest szersza niż samo „T5 lub BART”.

### Najważniejsze warianty

- **T5** – klasyczny model text-to-text; bardzo dobry punkt odniesienia architektonicznego.
- **FLAN-T5** – wariant T5 po instruction tuning; zwykle lepiej reaguje na naturalne polecenia zero-shot.
- **mT5** – wielojęzyczna odmiana T5 trenowana na **101 językach**.
- **ByT5** – wariant działający bez klasycznego tokenizera, bezpośrednio na bajtach UTF-8; bywa bardziej odporny na literówki i szum.
- **LongT5** – odmiana do długich sekwencji; oficjalna dokumentacja opisuje obsługę wejść nawet do **16 384 tokenów**.
- **T5Gemma** – nowsza rodzina encoder-decoder oparta na linii Gemma, dostępna także w wariantach instruction-tuned.
- **BART** – klasyczny model seq2seq łączący encoder typu BERT z autoregresywnym decoderem; nadal bardzo użyteczny jako mocny baseline do generacji warunkowej.

### Wniosek praktyczny

Aktualnie **czyste T5** częściej służy jako punkt odniesienia architektonicznego, a **FLAN-T5** jest zwykle wygodniejszym wyborem do demonstracji i zadań instrukcyjnych.  
**BART** pozostaje dobrym punktem wyjścia do summarization, zwłaszcza gdy używany jest checkpoint już dostrojony do tego zadania.


### Ręczne wczytywanie modelu seq2seq

Do pierwszego przykładu zostanie użyty `google/flan-t5-small`, ponieważ:

- architektura pozostaje zgodna z rodziną T5,
- model jest mały i wygodny do demonstracji,
- instruction tuning sprawia, że lepiej reaguje na naturalne polecenia niż klasyczne `t5-small`.

Warto zwrócić uwagę na trzy elementy:

- `AutoTokenizer` dobiera właściwy tokenizer,
- `AutoModelForSeq2SeqLM` wczytuje model przygotowany do generowania wyjścia na podstawie wejścia,
- `generate()` tworzy **osobną sekwencję wyjściową**, a nie tylko kontynuuje wejściowy prompt.


In [7]:
model_id = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

text = "Summarize in one sentence: Transformers process sequences with attention instead of recurrence."
inputs = tokenizer(text, return_tensors="pt")

print("Klucze wejścia:", inputs.keys())
print("Kształt input_ids:", inputs["input_ids"].shape)

outputs = model.generate(**inputs, max_new_tokens=30)
decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nWynik:")
print(decoded)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Klucze wejścia: KeysView({'input_ids': tensor([[12198,  1635,  1737,    16,    80,  7142,    10, 31220,     7,   433,
          5932,     7,    28,  1388,  1446,    13,     3,    60,  3663,    52,
          1433,     5,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])})
Kształt input_ids: torch.Size([1, 23])

Wynik:
Watch the recurrence sequences.


To zwraca dane wejściowe w postaci słownika, np.:

- `input_ids` – identyfikatory tokenów,
- `attention_mask` – informacja, które pozycje są rzeczywistym tekstem.

Następnie model generuje **nową sekwencję wyjściową**.  
To odróżnia ten tryb pracy od klasycznych modeli decoder-only, które zwykle po prostu dopisują ciąg po lewej stronie.


## Zmiana API w `transformers` 5.x

W starszych materiałach bardzo często pojawiały się konstrukcje:

- `pipeline("summarization")`
- `pipeline("text2text-generation")`

W nowszych wersjach biblioteki nie należy już opierać notebooka na tych skrótach. W praktyce najstabilniejsze są dziś dwa podejścia:

1. **`AutoTokenizer + AutoModelForSeq2SeqLM + generate()`** – najlepsze do nauki modeli encoder-decoder, pełnej kontroli nad wejściem i parametrów generacji,
2. **`pipeline("text-generation")` z nowoczesnym modelem chatowym** – dobre do ogólnego streszczania, QA i zadań wykonywanych instrukcją.

Ponieważ celem tego materiału jest zrozumienie rodzin **T5 / FLAN-T5 / BART**, dalsze przykłady korzystają z podejścia pierwszego.


## 1. Summarization – klasyczny use case dla BART

Summarization bardzo dobrze pokazuje sens architektury encoder-decoder:

- wejście trzeba przeczytać w całości,
- wynik trzeba wygenerować jako nowy tekst,
- samo „rozumienie” albo samo „dopisywanie” nie wystarcza.

Do demonstracji zostanie użyty `facebook/bart-large-cnn`, czyli klasyczny checkpoint BART dostrojony do summarization na zbiorze **CNN/DailyMail**.

### Ważna uwaga

- część starszych model cards nadal pokazuje przykłady z `pipeline("summarization")`,
- w aktualnym API bezpieczniej używać bezpośrednio `generate()`,
- ten model jest **anglojęzyczny**,
- najlepiej działa na tekstach zbliżonych do stylu newsowego,
- nie należy oczekiwać, że będzie najlepszym wyborem dla dokumentów domenowych, bardzo długich materiałów lub tekstów po polsku.

W takich sytuacjach częściej sięga się po model wielojęzyczny, model długiego kontekstu albo nowszy model instruction-tuned.


In [8]:
summary_model_id = "facebook/bart-large-cnn"

summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_id)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(summary_model_id).to(device)
summary_model.eval()

article = """
The company announced a broad restructuring plan after several quarters of slowing revenue growth.
Executives said the new strategy will focus on enterprise products, international expansion and cost control.
The plan includes layoffs in non-core teams, additional investment in cloud infrastructure and a reorganization of product leadership.
According to management, demand from large business customers remains strong, but consumer spending has weakened in key regions.
Analysts reacted cautiously, noting that the company had already promised efficiency improvements earlier this year.
During the earnings call, the chief executive said the restructuring should improve margins over the next four quarters.
The board also approved a new stock repurchase program, which was interpreted by investors as a signal of confidence.
Shares rose in after-hours trading, although several analysts warned that execution risks remain significant.
"""

inputs = summary_tokenizer(
    article,
    return_tensors="pt",
    truncation=True,
    max_length=1024
).to(device)

with torch.inference_mode():
    summary_ids = summary_model.generate(
        **inputs,
        max_new_tokens=48,
        min_new_tokens=18,
        num_beams=6,
        length_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True
    )

print(summary_tokenizer.decode(summary_ids[0], skip_special_tokens=True))

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Both `max_new_tokens` (=48) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=18) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The company announced a broad restructuring plan after several quarters of slowing revenue growth. The plan includes layoffs in non-core teams, additional investment in cloud infrastructure and a reorganization of product leadership.


In [9]:
summary_ids

tensor([[    2,     0,   133,   138,   585,    10,  4007,  8183,   563,    71,
           484,  5666,     9,  8464,   903,   434,     4,    20,   563,  1171,
         22788,    11,   786,    12,  7293,   893,     6,   943,   915,    11,
          3613,  2112,     8,    10, 22208,  1938,     9,  1152,  1673,     4,
             2]], device='cuda:0')

### Ćwiczenie

Należy uruchomić poniższy kod i sprawdzić, jak model zachowuje się dla trzech różnych tekstów:

- technicznego,
- publicystycznego,
- bardziej chaotycznego lub z powtórzeniami.

Następnie warto ocenić:

1. czy model zachowuje najważniejsze informacje,
2. czy skraca tekst sensownie, czy tylko usuwa fragmenty,
3. kiedy summary brzmi naturalnie, a kiedy zbyt mechanicznie.


In [10]:
_seq2seq_cache = {}

def get_seq2seq_components(model_id):
    if model_id not in _seq2seq_cache:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
        model.eval()
        _seq2seq_cache[model_id] = (tokenizer, model)
    return _seq2seq_cache[model_id]

def run_summarization(model_id, text, max_new_tokens=60, min_new_tokens=20, max_input_tokens=1024):
    tokenizer, model = get_seq2seq_components(model_id)
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    ).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=4, # wybór kandydatów do wygenerowania
            no_repeat_ngram_size=3, # zabrania powtarzania tych samych 3-wyrazowych / 3-tokenowych sekwencji
            early_stopping=True # przy beam search generacja kończy się wcześniej, gdy wyszukiwanie uzna, że znaleziono już wystarczająco zakończonych dobrych kandydatów
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

texts = [
    '''
    Large language models can support business analysis, but their output is only useful when
    it is controlled, verified and aligned with the goal of the task. In many organizations,
    the biggest problem is not generation itself, but the mismatch between a fluent answer and
    the factual requirements of the user.
    ''',
    '''
    Large language models can support business analysis, but their output is only useful when
    controlled, verified and aligned with the goal of the task. In many organizations,
    the biggest problem is not generation itself, but the mismatch between a fluent answer and
    the factual requirements of the user.
    '''

    '''
    The city announced a new transport policy focused on reducing traffic and increasing the use
    of public transport. Officials explained that the long-term goal is lower pollution, shorter
    travel times and better accessibility for residents living farther from the city center.
    ''',
    '''
    Transformers are powerful. They are used in NLP. They are also used in summarization.
    Summarization is useful because long texts are hard to read. Long texts are sometimes repetitive.
    Repetition can make the source noisy, and noisy input can make summaries less stable.
    '''
]

for i, text in enumerate(texts, start=1):
    print(f"\n=== PRZYKŁAD {i} ===")
    print(run_summarization("facebook/bart-large-cnn", text))



=== PRZYKŁAD 1 ===


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=20) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=20) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Large language models can support business analysis, but their output is only useful when controlled, verified and aligned with the goal of the task. In many organizations, the biggest problem is not generation itself, but the mismatch between a fluent answer and the factual requirements of the user.

=== PRZYKŁAD 2 ===


Both `max_new_tokens` (=60) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=20) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The city announced a new transport policy focused on reducing traffic and increasing the use of public transport. Officials explained that the long-term goal is lower pollution, shorter travel times and better accessibility.

=== PRZYKŁAD 3 ===
Transformers are used in NLP. They are also used in summarization. Long texts are sometimes repetitive. Repetition can make the source noisy.


## 2. Text2Text – najczystsza demonstracja idei T5

Rodzina T5 została zaprojektowana tak, aby wiele różnych zadań dało się zapisać w jednym formacie:

**tekst wejściowy → tekst wyjściowy**

To podejście pozostaje bardzo ważne, ponieważ ułatwia:

- traktowanie klasyfikacji jako generowania etykiety,
- traktowanie tłumaczenia jako generowania tłumaczenia,
- traktowanie streszczania jako generowania podsumowania,
- traktowanie QA jako generowania odpowiedzi.

W aktualnym API najczytelniej pokazuje się to już nie przez `pipeline("text2text-generation")`, tylko przez bezpośrednie wywołanie `generate()` na modelu encoder-decoder.

W praktyce instruction-tuned warianty, takie jak **FLAN-T5**, zwykle lepiej radzą sobie z naturalnie zapisanym poleceniem niż surowe checkpointy T5.


In [11]:
model_cache = {}

def get_seq2seq_components(model_id):
    if model_id not in model_cache:
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)
        model.eval()
        model_cache[model_id] = (tokenizer, model)
    return model_cache[model_id]


def run_seq2seq(
    model_id,
    prompt,
    max_new_tokens=60,
    max_input_tokens=1024,
    num_beams=4,
    no_repeat_ngram_size=2
):
    tokenizer, model = get_seq2seq_components(model_id)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    ).to(device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_length=None,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            no_repeat_ngram_size=no_repeat_ngram_size,
            early_stopping=True
        )

    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


summary_model_id = "facebook/bart-large-cnn"
translation_model_id = "Helsinki-NLP/opus-mt-en-de"
instruction_model_id = "google/flan-t5-base"

prompts = [
    "summarize in one short sentence: Machine learning models can overfit when they memorize the training data instead of learning general patterns.",
    "translate English to German, output only the translation: Transformers are useful for sequence-to-sequence tasks.",
    "rewrite in simpler English, keep the meaning, output one sentence: Large language models may sound convincing even when the answer is incomplete.",
    "answer in one sentence only: What does cross-attention do in an encoder-decoder model?"
]

article = """
The company announced a broad restructuring plan after several quarters of slowing revenue growth.
Executives said the new strategy will focus on enterprise products, international expansion and cost control.
The plan includes layoffs in non-core teams, additional investment in cloud infrastructure and a reorganization of product leadership.
According to management, demand from large business customers remains strong, but consumer spending has weakened in key regions.
Analysts reacted cautiously, noting that the company had already promised efficiency improvements earlier this year.
During the earnings call, the chief executive said the restructuring should improve margins over the next four quarters.
The board also approved a new stock repurchase program, which investors interpreted as a sign of confidence.
Shares rose in after-hours trading, although several analysts warned that execution risks remain significant.
"""

print("STRESZCZENIE:")
print(run_seq2seq(summary_model_id, article, max_new_tokens=128, no_repeat_ngram_size=3))

text_en = "Transformers are useful for sequence-to-sequence tasks."

print("TŁUMACZENIE EN -> DE:")
print(run_seq2seq(translation_model_id, text_en, max_new_tokens=128, no_repeat_ngram_size=0))

prompt = (
    "rewrite in simpler English: "
    "In addition to neuronal and synaptic state, SNNs incorporate the concept of time into their operating model. The idea is that neurons in the SNN do not transmit information at each propagation cycle (as it happens with typical multi-layer perceptron networks), but rather transmit information only when a membrane potential."
)

print("PODEJŚCIE INSTRUKCYJNE:")
print(run_seq2seq(instruction_model_id, prompt, max_new_tokens=128, no_repeat_ngram_size=2))

STRESZCZENIE:


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

The company announced a broad restructuring plan after several quarters of slowing revenue growth. The plan includes layoffs in non-core teams, additional investment in cloud infrastructure and a reorganization of product leadership. The board also approved a new stock repurchase program, which investors interpreted as a sign of confidence.
TŁUMACZENIE EN -> DE:


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In addition to neuronal and synaptic state, SNNs incorporate the concept of time into their operating model.


### Ćwiczenie

Należy dopisać własne cztery prompty:

- jeden do summarization,
- jeden do translation,
- jeden do uproszczenia tekstu,
- jeden do odpowiedzi na pytanie.

Następnie warto odpowiedzieć:

1. które zadania FLAN-T5 wykonuje najlepiej,
2. czy model rozumie polecenie równie dobrze w każdej formie,
3. jak bardzo wynik zależy od precyzji promptu.


## 3. Różnica między `AutoModelForSeq2SeqLM` a `AutoModelForCausalLM`

To rozróżnienie pozostaje kluczowe.

### `AutoModelForCausalLM`
Model typu decoder-only.  
Głównym celem jest **kontynuowanie sekwencji**.

### `AutoModelForSeq2SeqLM`
Model encoder-decoder.  
Głównym celem jest **przekształcenie jednej sekwencji w drugą**.

Na poziomie API oba modele potrafią „generować”, ale logika pracy jest inna.

W praktyce:

- `TextGenerationPipeline` dominuje w pracy z nowoczesnymi modelami chatowymi,
- rodzina encoder-decoder nadal pozostaje bardzo użyteczna wtedy, gdy potrzebna jest bardziej kontrolowana transformacja wejścia w wyjście.


In [12]:
seq2seq_model_id = "google/flan-t5-small"
causal_model_id = "distilgpt2"

seq2seq_tokenizer = AutoTokenizer.from_pretrained(seq2seq_model_id)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(seq2seq_model_id).to(device)
seq2seq_model.eval()

causal_tokenizer = AutoTokenizer.from_pretrained(causal_model_id)
causal_tokenizer.pad_token = causal_tokenizer.eos_token
causal_model = AutoModelForCausalLM.from_pretrained(causal_model_id).to(device)
causal_model.eval()

seq2seq_prompt = "Summarize in one sentence: Deep learning models can process text, images and audio, but they still require evaluation."
causal_prompt = "Deep learning models can"

seq2seq_inputs = seq2seq_tokenizer(seq2seq_prompt, return_tensors="pt").to(device)
causal_inputs = causal_tokenizer(causal_prompt, return_tensors="pt").to(device)

with torch.inference_mode():
    seq2seq_output = seq2seq_model.generate(**seq2seq_inputs, max_new_tokens=20)
    causal_output = causal_model.generate(
        **causal_inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=causal_tokenizer.eos_token_id
    )

print("SEQ2SEQ OUTPUT:")
print(seq2seq_tokenizer.decode(seq2seq_output[0], skip_special_tokens=True))

print("\nCAUSAL OUTPUT:")
print(causal_tokenizer.decode(causal_output[0], skip_special_tokens=True))


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

SEQ2SEQ OUTPUT:
Deep learning models can be used to learn and analyze data.

CAUSAL OUTPUT:
Deep learning models can be used to create a new learning model.













### Interpretacja

Wynik modelu seq2seq zwykle będzie próbą wykonania zadania wpisanego w poleceniu, np. podsumowania.  
Wynik modelu causal będzie raczej kontynuacją rozpoczętego fragmentu.

To oznacza, że:

- `AutoModelForCausalLM` lepiej pasuje do generacji otwartej, completion i chatu,
- `AutoModelForSeq2SeqLM` lepiej pasuje do transformacji tekstu wejściowego w tekst wyjściowy.

Aktualnie praktyka produkcyjna bardzo często łączy oba podejścia:  
model decoder-only obsługuje interakcję otwartą, a model seq2seq wykonuje zadania bardziej kontrolowane.


### Mini-ćwiczenie

Należy uruchomić powyższy kod dla trzech par promptów:

- technicznych,
- codziennych,
- mieszanych językowo.

Następnie warto ocenić:

1. kiedy model causal „odpływa” w generację niezgodną z celem,
2. kiedy model seq2seq daje bardziej kontrolowany wynik,
3. czy sam prompt wystarcza, aby decoder-only zachowywał się jak model zadaniowy.


## 4. Porównanie klasycznego T5 i FLAN-T5

To porównanie dobrze pokazuje jedną z najważniejszych aktualizacji względem starszych materiałów:

- **`t5-small`** jest świetnym przykładem historycznej architektury text-to-text,
- **`google/flan-t5-small`** pokazuje, co daje instruction tuning.

W praktyce często bardziej sensowne jest porównanie **T5 vs FLAN-T5** niż T5 vs BART w trybie zero-shot, ponieważ różnica dobrze pokazuje wpływ dostrojenia instrukcyjnego.


In [13]:
comparison_prompts = [
    {
        "task": "kanoniczne summarization",
        "prompt": "summarize: Attention lets a model connect distant tokens without recurrence."
    },
    {
        "task": "instrukcyjne tłumaczenie",
        "prompt": "Translate the following sentence into German: Encoder-decoder models transform one text into another."
    },
    {
        "task": "krótkie QA",
        "prompt": "Answer briefly: Why is cross-attention useful in a seq2seq model?"
    }
]

rows = []

for item in comparison_prompts:
    prompt = item["prompt"]
    rows.append({
        "task": item["task"],
        "prompt": prompt,
        "t5_small": run_seq2seq("t5-small", prompt, max_new_tokens=50),
        "flan_t5_small": run_seq2seq("google/flan-t5-small", prompt, max_new_tokens=50)
    })

pd.DataFrame(rows)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


,task,prompt,t5_small,flan_t5_small
0,kanoniczne summarization,summarize: Attention lets a model connect dist...,Attention lets a model connect distant tokens ...,attention lets a model connect distant tokens ...
1,instrukcyjne tłumaczenie,Translate the following sentence into German: ...,Die folgenden Satz: Encoder-Dekodatormodelle t...,Die Modelle von encoder-decoderes transformier...
2,krótkie QA,Answer briefly: Why is cross-attention useful ...,Answer briefly: Why is cross-attention useful ...,cross-attention is useful in a seq2seq model


### Interpretacja

Przy analizie wyników warto zwrócić uwagę na to, że:

- klasyczne T5 często działa najlepiej wtedy, gdy prompt ma formę zgodną z dawnym stylem zadań T5,
- FLAN-T5 zwykle lepiej reaguje na bardziej naturalne instrukcje,
- instruction tuning nie zmienia samej architektury encoder-decoder, ale istotnie poprawia użyteczność modelu zero-shot.

To jest jedna z najważniejszych praktycznych różnic między materiałem „historycznym” a podejściem nowszym.


## 5. Porównanie FLAN-T5 i BART w summarization

To porównanie odpowiada bardziej współczesnej praktyce:

- **FLAN-T5** reprezentuje model bardziej uniwersalny i instrukcyjny,
- **BART Large CNN** reprezentuje klasyczny model mocno związany z konkretnym zadaniem summarization.

Nie jest to porównanie „który model jest absolutnie lepszy”, tylko porównanie **dwóch różnych filozofii użycia**:

1. model bardziej ogólny,  
2. model mocno dostrojony do konkretnego zadania.


In [14]:
comparison_examples = [
    {
        "text": "Transformers made NLP training more parallel by replacing recurrence with attention. This improved scalability and enabled stronger models for understanding and generation.",
        "reference": "Transformers improved scalability in NLP by replacing recurrence with attention."
    },
    {
        "text": "Organizations increasingly use AI systems to summarize reports, emails and meetings. The main challenge is not fluency alone, but preserving facts, intent and the most useful details.",
        "reference": "AI summarization is useful in organizations, but factual accuracy and retention of key details remain crucial."
    }
]

flan_predictions = []
bart_predictions = []

for item in comparison_examples:
    text = item["text"]
    flan_pred = run_seq2seq(
        "google/flan-t5-small",
        f"Summarize in one sentence: {text}",
        max_new_tokens=40
    )
    bart_pred = run_summarization(
        "facebook/bart-large-cnn",
        text,
        max_new_tokens=40,
        min_new_tokens=10
    )
    flan_predictions.append(flan_pred)
    bart_predictions.append(bart_pred)

comparison_df = pd.DataFrame({
    "input": [item["text"] for item in comparison_examples],
    "reference": [item["reference"] for item in comparison_examples],
    "flan_t5_small": flan_predictions,
    "bart_large_cnn": bart_predictions
})

comparison_df

Both `max_new_tokens` (=40) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=40) and `max_length`(=142) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `min_new_tokens` (=10) and `min_length`(=56) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,input,reference,flan_t5_small,bart_large_cnn
0,Transformers made NLP training more parallel b...,Transformers improved scalability in NLP by re...,Transformers has been able to increase the num...,Transformers made NLP training more parallel b...
1,Organizations increasingly use AI systems to s...,"AI summarization is useful in organizations, b...",Learn how to use AI to summarize reports and e...,Organizations increasingly use AI systems to s...


## 6. Prosta ewaluacja jakości: ręczna i automatyczna

Aktualnie nadal nie wystarcza spojrzenie wyłącznie na płynność języka.  
Ocena powinna łączyć co najmniej dwa poziomy:

### Ocena ręczna
Najprostsze kryteria:

- **zgodność z wejściem** – czy summary rzeczywiście wynika z tekstu,
- **zwięzłość** – czy wynik jest krótszy i nadal sensowny,
- **pokrycie kluczowych informacji** – czy zachowano najważniejsze elementy,
- **halucynacje** – czy dopisano coś, czego nie było w wejściu.

### Ocena automatyczna
Do szybkiego benchmarku można użyć np. **ROUGE**.  
Trzeba jednak pamiętać, że wysoki wynik metryki nie gwarantuje poprawności semantycznej i nie wykrywa wszystkich halucynacji.


In [15]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=acc0c02616bdc7a4cbdda7ccc92f083ca3d4994e17fdb3d66e4a1e1bb277635d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [16]:
rouge = evaluate.load("rouge")

flan_scores = rouge.compute(
    predictions=flan_predictions,
    references=[item["reference"] for item in comparison_examples]
)

bart_scores = rouge.compute(
    predictions=bart_predictions,
    references=[item["reference"] for item in comparison_examples]
)

pd.DataFrame([
    {"model": "google/flan-t5-small", **flan_scores},
    {"model": "facebook/bart-large-cnn", **bart_scores}
])

,model,rouge1,rouge2,rougeL,rougeLsum
0,google/flan-t5-small,0.196923,0.000000,0.156923,0.156923
1,facebook/bart-large-cnn,0.462791,0.178571,0.349612,0.349612


### Karta oceny jakości

Dla każdego przykładu warto ocenić oba modele w skali 1–5:

- zgodność z wejściem,
- płynność językową,
- zachowanie najważniejszych informacji,
- ogólną użyteczność wyniku.


## 7. Zadanie

Przygotować w Google Colab prosty interfejs webowy w Gradio do tłumaczenia tekstu. Użytkownik ma wpisać tekst w wybranym języku, wybrać język docelowy z listy, a aplikacja ma automatycznie dobrać odpowiedni model tłumaczeniowy.

Należy:

- uzupełnić słownik mapujący język docelowy na identyfikator modelu,
- wyszukać odpowiednie modele w rodzinie Helsinki-NLP / OPUS-MT na Hugging Face,
- zaimplementować ładowanie tokenizera i modelu,
- uzupełnić funkcję tłumaczącą tekst,
- połączyć komponenty Gradio tak, aby zmiana języka aktualizowała nazwę modelu, - a kliknięcie przycisku uruchamiało tłumaczenie.

<img src="https://i.ibb.co/Fb1svQVG/image.png" width="100%"/>


In [17]:
!pip -q install gradio transformers sentencepiece

In [18]:
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

DEVICE: cuda


In [20]:
# UZUPEŁNIĆ:
# dobrać modele z rodziny Helsinki-NLP / OPUS-MT
# tak, aby wybór języka oznaczał wybór odpowiedniego modelu tłumaczeniowego
LANG_TO_MODEL = {
    ...
}

# cache, żeby model nie ładował się od nowa przy każdym kliknięciu
model_cache = {}

def get_translation_components(language):
    # UZUPEŁNIĆ:
    # 1. odczytać model_id ze słownika LANG_TO_MODEL
    # 2. jeśli model nie jest w cache:
    #    - załadować tokenizer
    #    - załadować model
    #    - przenieść model na device
    #    - ustawić model.eval()
    #    - zapisać do cache
    # 3. zwrócić model_id, tokenizer, model
    pass

def update_model_name(language):
    # UZUPEŁNIĆ:
    # zwrócić nazwę modelu odpowiadającego wybranemu językowi
    pass

def translate_text(text, language):
    text = (text or "").strip()

    # UZUPEŁNIĆ:
    # pobrać model_id, tokenizer i model
    pass

    if not text:
        return "", model_id

    # UZUPEŁNIĆ:
    # przygotować tokenizację wejścia
    # inputs = ...

    with torch.inference_mode():
        # UZUPEŁNIĆ:
        # wygenerować tłumaczenie metodą generate()
        # output_ids = ...
        pass

    # UZUPEŁNIĆ:
    # zdekodować wynik do tekstu
    # translation = ...

    return translation, model_id

with gr.Blocks() as demo:
    gr.Markdown("# Tłumacz tekstu")
    gr.Markdown("Wejście: tekst w wybranym języku. Wybór języka automatycznie wybiera model tłumaczący.")

    with gr.Row():
        language = gr.Dropdown(
            choices=list(LANG_TO_MODEL.keys()),
            value=list(LANG_TO_MODEL.keys())[0] if len(LANG_TO_MODEL) > 0 else None,
            label="Język docelowy"
        )
        model_name = gr.Textbox(
            value="",
            label="Wybrany model",
            interactive=False
        )

    input_text = gr.Textbox(
        label="Tekst do tłumaczenia (EN)",
        lines=6,
        placeholder="Enter English text here..."
    )

    output_text = gr.Textbox(
        label="Tłumaczenie",
        lines=6
    )

    translate_btn = gr.Button("Tłumacz")

    # UZUPEŁNIĆ:
    # po zmianie języka zaktualizować nazwę modelu
    # language.change(...)

    # UZUPEŁNIĆ:
    # po kliknięciu przycisku uruchomić tłumaczenie
    # translate_btn.click(...)

    gr.Examples(
        examples=[
            ["Transformers are useful for sequence-to-sequence tasks."],
            ["The company announced a broad restructuring plan after several quarters of slowing revenue growth."],
            ["Machine learning models can overfit when they memorize the training data instead of learning general patterns."]
        ],
        inputs=input_text
    )

demo.launch(share=True)

AttributeError: 'set' object has no attribute 'keys'

## Źródła i dokumentacja

Oficjalna dokumentacja i źródła użyte przy aktualizacji materiału:

- [Transformers v5 Migration Guide – Hugging Face / GitHub](https://github.com/huggingface/transformers/blob/main/MIGRATION_GUIDE_V5.md)
- [Transformers releases – usunięcie starych przykładów pipeline dla summarization/translation](https://github.com/huggingface/transformers/releases)
- [Pipeline tutorial – aktualne zadania i sposób użycia pipeline](https://huggingface.co/docs/transformers/pipeline_tutorial)
- [Pipelines – Hugging Face Transformers](https://huggingface.co/docs/transformers/en/main_classes/pipelines)
- [T5 – Hugging Face Transformers](https://huggingface.co/docs/transformers/model_doc/t5)
- [FLAN-T5 – Hugging Face Transformers](https://huggingface.co/docs/transformers/model_doc/flan-t5)
- [mT5 – Hugging Face Transformers](https://huggingface.co/docs/transformers/model_doc/mt5)
- [ByT5 – Hugging Face Transformers](https://huggingface.co/docs/transformers/model_doc/byt5)
- [LongT5 – Hugging Face Transformers](https://huggingface.co/docs/transformers/model_doc/longt5)
- [T5Gemma – Hugging Face Transformers](https://huggingface.co/docs/transformers/model_doc/t5gemma)
- [BART – Hugging Face Transformers](https://huggingface.co/docs/transformers/model_doc/bart)
- [facebook/bart-large-cnn – model card](https://huggingface.co/facebook/bart-large-cnn)
- [HELSKINKI](https://huggingface.co/Helsinki-NLP)

Najważniejsza konsekwencja praktyczna dla tego notebooka jest prosta: demonstracje seq2seq pokazano przez `generate()`, a nie przez historyczne pipeline’y `summarization` i `text2text-generation`.
